<a href="https://colab.research.google.com/github/thehouseisonfire/t1/blob/main/T2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Estudante/Matrícula**: Pedro Dariva/2111100009

**Estudante/Matrícula**: Pedro Spegiorin/2111100026

# Trabalho T2 - Redes Neurais Convolucionais

Este trabalho tem como objetivo a compreensão de como se dá o processo de treinamento de uma rede neural artificial (RNA) do tipo convolucional. A execução deste exercício permitirá aos estudantes o domínio dos métodos de regularização necessários para o treinamento e, posteriormente, de teste de uma RNA utilizando um dataset pré-processado.

## 1. Instruções

O estudante deve treinar uma RNA com camadas convolucionais e densas, no estilo feedforward, para uma tarefa de classificação utilizando dataset pré-processado conforme especificado abaixo.

Para cada tentativa de construção de arquitetura, mantenha um registro da tentativa e resultado obtido pela rede. Indique qual a lógica utilizada para criação da arquitetura e qual a percepção do resultado em comparação com tentativas anteriores.

Para este trabalho, somente serão permitidas somente as seguintes bibliotecas `python` para o treinamento da RNA:

- `numpy`
- `scipy`
- `tensorflow`
- `scikit-learn`

Para visualização e criação de gráficos, também serão permitidas

- `matplotlib`
- `seaborn`
- `plotly`
- `yellowbrick`

Caso a utilização de quaisquer outras bibliotecas se faça necessária, uma consulta prévia deve ser feita ao professor.


## Entrega

O presente trabalho pode ser feito individualmente ou em dupla. Caso optem pelo trabalho em dupla, o cabeçalho do documento deve indicar quais são os componentes e suas matrículas.

A entrega deverá ser feita através do envio de um arquivo `ZIP` contendo o trabalho dos estudantes, no SIGAA. A data limite para entrega deste trabalho é **22/10/2023, 23h59m**.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.config.list_physical_devices())


In [ ]:
(train_data, test_data), dataset_info = tfds.load(
    "cifar10",
    split=("train", "test"),
    with_info=True,
    batch_size=-1,
)
train_data = tfds.as_numpy(train_data)
test_data = tfds.as_numpy(test_data)

class_names = dataset_info.features["label"].names
num_classes = dataset_info.features["label"].num_classes


In [ ]:
x_train, y_train = train_data["image"], train_data["label"]
x_test, y_test = test_data["image"], test_data["label"]


In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(15, 8))
for image, label, axis in zip(x_train[:50], y_train[:50], axes.flat):
    axis.imshow(image)
    axis.set_title(class_names[label], fontsize=8)
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.16,
    stratify=y_train,
    random_state=SEED,
)

x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(f"{len(x_train)} exemplos de treino")
print(f"{len(x_val)} exemplos de validação")
print(f"{len(x_test)} exemplos de teste")


## Inclua seu código abaixo desta célula

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.Input(shape=x_train.shape[1:]),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(30, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.RMSprop(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
    ],
)
model.summary()

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=10,
    validation_data=(x_val, y_val),
)

test_loss, test_accuracy, test_top_5 = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test top-5 accuracy: {test_top_5:.2%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"], label="treino")
ax1.plot(history.history["val_accuracy"], label="validação")
ax1.set(title="Acurácia", xlabel="Época", ylabel="Acurácia")
ax1.legend()

ax2.plot(history.history["loss"], label="treino")
ax2.plot(history.history["val_loss"], label="validação")
ax2.set(title="Loss", xlabel="Época", ylabel="Loss")
ax2.legend()
plt.tight_layout()
plt.show()

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    xticks_rotation=45,
)
plt.tight_layout()
plt.show()

print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}")
print("\nMétricas por classe:")
print(classification_report(y_test, y_pred, target_names=class_names))
